# 2. 전처리·피처 구성 및 누수 점검

서울시 따릉이 일별 수요 예측. [요구사항](../../docs/ml/REQUIREMENTS.md).

In [1]:
from pathlib import Path
import sys, json, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
from src.ml.features import FEATURES, load_daily, make_features
from sklearn.base import clone
from sklearn.inspection import permutation_importance

plt.rcParams.update({"figure.figsize": (10, 4), "font.family": "Malgun Gothic", "axes.unicode_minus": False})
OUT = ROOT / "outputs/ml"
OUT.mkdir(parents=True, exist_ok=True)
LOCAL = ROOT / "data/processed"
LOCAL.mkdir(parents=True, exist_ok=True)
daily = load_daily(ROOT)
frame = make_features(daily)


In [2]:
quality = pd.DataFrame([{
    "raw_rows": len(daily), "feature_rows": len(frame), "warmup_excluded": len(daily) - len(frame),
    "duplicate_dates": int(daily.date.duplicated().sum()),
    "missing_days": int(len(pd.date_range(daily.date.min(), daily.date.max())) - len(daily)),
    "negative_targets": int((daily.rentals < 0).sum()),
}])
quality.to_csv(OUT / "02_quality.csv", index=False)
display(quality)
display(pd.DataFrame({"feature": FEATURES, "available": ["D-2", "D-7", "D-14", "D-8~D-2", "D-29~D-2"] + ["D일 달력"] * 5}))
changed = daily.copy()
cutoff = pd.Timestamp("2025-08-01")
changed.loc[changed.date >= cutoff - pd.Timedelta(days=1), "rentals"] += 1_000_000
altered = make_features(changed)
pd.testing.assert_frame_equal(frame.loc[frame.date <= cutoff, FEATURES], altered.loc[altered.date <= cutoff, FEATURES])
print("누수 점검 통과: D-1 이후의 대여건수 변경이 D일 피처에 영향을 주지 않음")


,raw_rows,feature_rows,warmup_excluded,duplicate_dates,missing_days,negative_targets
0,546,517,29,0,0,0


,feature,available
0,lag_2,D-2
1,lag_7,D-7
2,lag_14,D-14
3,rolling_7,D-8~D-2
4,rolling_28,D-29~D-2
5,dow_sin,D일 달력
6,dow_cos,D일 달력
7,year_sin,D일 달력
8,year_cos,D일 달력
9,weekend,D일 달력


누수 점검 통과: D-1 이후의 대여건수 변경이 D일 피처에 영향을 주지 않음


- 원본 546일에서 이동통계 준비 구간 29일을 제외해 517일을 구성함.
- 중복·누락 날짜와 음수 타깃이 없으며 날짜를 건너뛴 지연 피처 생성을 방지함.
- D-1 이후 값 변경에도 D일 피처가 동일해 설계한 정보 시점 조건을 충족함.